# 03 — Train Acgan / 生成对抗网络

**Chapter 19 — File 3 of 4 / 第19章 — 第3个文件（共4个）**

---

## Summary / 总结

This script demonstrates **example of fitting an auxiliary classifier gan (ac-gan) on fashion mnsit**.

本脚本演示 **example of fitting an auxiliary classifier gan (ac-gan) on fashion mnsit**。

---
## Step 1 — example of fitting an auxiliary classifier gan (ac-gan) on fashion mnsit

In [ ]:
from numpy import zeros
from numpy import ones
from numpy import expand_dims
from numpy.random import randn
from numpy.random import randint
from keras.datasets.fashion_mnist import load_data
from keras.optimizers import Adam
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Reshape
from keras.layers import Flatten
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.layers import BatchNormalization
from keras.layers import Dropout
from keras.layers import Embedding
from keras.layers import Activation
from keras.layers import Concatenate
from keras.initializers import RandomNormal
from matplotlib import pyplot

---
## Step 2 — define the standalone discriminator model

In [ ]:
def define_discriminator(in_shape=(28,28,1), n_classes=10):

---
## Step 3 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 4 — image input

In [ ]:
in_image = Input(shape=in_shape)

---
## Step 5 — downsample to 14x14

In [ ]:
fe = Conv2D(32, (3,3), strides=(2,2), padding='same', kernel_initializer=init)(in_image)
	fe = LeakyReLU(alpha=0.2)(fe)
	fe = Dropout(0.5)(fe)

---
## Step 6 — normal

In [ ]:
fe = Conv2D(64, (3,3), padding='same', kernel_initializer=init)(fe)
	fe = BatchNormalization()(fe)
	fe = LeakyReLU(alpha=0.2)(fe)
	fe = Dropout(0.5)(fe)

---
## Step 7 — downsample to 7x7

In [ ]:
fe = Conv2D(128, (3,3), strides=(2,2), padding='same', kernel_initializer=init)(fe)
	fe = BatchNormalization()(fe)
	fe = LeakyReLU(alpha=0.2)(fe)
	fe = Dropout(0.5)(fe)

---
## Step 8 — normal

In [ ]:
fe = Conv2D(256, (3,3), padding='same', kernel_initializer=init)(fe)
	fe = BatchNormalization()(fe)
	fe = LeakyReLU(alpha=0.2)(fe)
	fe = Dropout(0.5)(fe)

---
## Step 9 — flatten feature maps

In [ ]:
fe = Flatten()(fe)

---
## Step 10 — real/fake output

In [ ]:
out1 = Dense(1, activation='sigmoid')(fe)

---
## Step 11 — class label output

In [ ]:
out2 = Dense(n_classes, activation='softmax')(fe)

---
## Step 12 — define model

In [ ]:
model = Model(in_image, [out1, out2])

---
## Step 13 — compile model

In [ ]:
opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss=['binary_crossentropy', 'sparse_categorical_crossentropy'], optimizer=opt)
	return model

---
## Step 14 — define the standalone generator model

In [ ]:
def define_generator(latent_dim, n_classes=10):

---
## Step 15 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 16 — label input

In [ ]:
in_label = Input(shape=(1,))

---
## Step 17 — embedding for categorical input

In [ ]:
li = Embedding(n_classes, 50)(in_label)

---
## Step 18 — linear multiplication

In [ ]:
n_nodes = 7 * 7
	li = Dense(n_nodes, kernel_initializer=init)(li)

---
## Step 19 — reshape to additional channel

In [ ]:
li = Reshape((7, 7, 1))(li)

---
## Step 20 — image generator input

In [ ]:
in_lat = Input(shape=(latent_dim,))

---
## Step 21 — foundation for 7x7 image

In [ ]:
n_nodes = 384 * 7 * 7
	gen = Dense(n_nodes, kernel_initializer=init)(in_lat)
	gen = Activation('relu')(gen)
	gen = Reshape((7, 7, 384))(gen)

---
## Step 22 — merge image gen and label input

In [ ]:
merge = Concatenate()([gen, li])

---
## Step 23 — upsample to 14x14

In [ ]:
gen = Conv2DTranspose(192, (5,5), strides=(2,2), padding='same', kernel_initializer=init)(merge)
	gen = BatchNormalization()(gen)
	gen = Activation('relu')(gen)

---
## Step 24 — upsample to 28x28

In [ ]:
gen = Conv2DTranspose(1, (5,5), strides=(2,2), padding='same', kernel_initializer=init)(gen)
	out_layer = Activation('tanh')(gen)

---
## Step 25 — define model

In [ ]:
model = Model([in_lat, in_label], out_layer)
	return model

---
## Step 26 — define the combined generator and discriminator model, for updating the generator

In [ ]:
def define_gan(g_model, d_model):

---
## Step 27 — make weights in the discriminator not trainable

In [ ]:
for layer in d_model.layers:
		if not isinstance(layer, BatchNormalization):
			layer.trainable = False

---
## Step 28 — connect the outputs of the generator to the inputs of the discriminator

In [ ]:
gan_output = d_model(g_model.output)

---
## Step 29 — define gan model as taking noise and label and outputting real/fake and label outputs

In [ ]:
model = Model(g_model.input, gan_output)

---
## Step 30 — compile model

In [ ]:
opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss=['binary_crossentropy', 'sparse_categorical_crossentropy'], optimizer=opt)
	return model

---
## Step 31 — load images

In [ ]:
def load_real_samples():

---
## Step 32 — load dataset

In [ ]:
(trainX, trainy), (_, _) = load_data()

---
## Step 33 — expand to 3d, e.g. add channels

In [ ]:
X = expand_dims(trainX, axis=-1)

---
## Step 34 — convert from ints to floats

In [ ]:
X = X.astype('float32')

---
## Step 35 — scale from [0,255] to [-1,1]

In [ ]:
X = (X - 127.5) / 127.5
	print(X.shape, trainy.shape)
	return [X, trainy]

---
## Step 36 — select real samples

In [ ]:
def generate_real_samples(dataset, n_samples):

---
## Step 37 — split into images and labels

In [ ]:
images, labels = dataset

---
## Step 38 — choose random instances

In [ ]:
ix = randint(0, images.shape[0], n_samples)

---
## Step 39 — select images and labels

In [ ]:
X, labels = images[ix], labels[ix]

---
## Step 40 — generate class labels

In [ ]:
y = ones((n_samples, 1))
	return [X, labels], y

---
## Step 41 — generate points in latent space as input for the generator

In [ ]:
def generate_latent_points(latent_dim, n_samples, n_classes=10):

---
## Step 42 — generate points in the latent space

In [ ]:
x_input = randn(latent_dim * n_samples)

---
## Step 43 — reshape into a batch of inputs for the network

In [ ]:
z_input = x_input.reshape(n_samples, latent_dim)

---
## Step 44 — generate labels

In [ ]:
labels = randint(0, n_classes, n_samples)
	return [z_input, labels]

---
## Step 45 — use the generator to generate n fake examples, with class labels

In [ ]:
def generate_fake_samples(generator, latent_dim, n_samples):

---
## Step 46 — generate points in latent space

In [ ]:
z_input, labels_input = generate_latent_points(latent_dim, n_samples)

---
## Step 47 — predict outputs

In [ ]:
images = generator.predict([z_input, labels_input])

---
## Step 48 — create class labels

In [ ]:
y = zeros((n_samples, 1))
	return [images, labels_input], y

---
## Step 49 — generate samples and save as a plot and save the model

In [ ]:
def summarize_performance(step, g_model, latent_dim, n_samples=100):

---
## Step 50 — prepare fake examples

In [ ]:
[X, _], _ = generate_fake_samples(g_model, latent_dim, n_samples)

---
## Step 51 — scale from [-1,1] to [0,1]

In [ ]:
X = (X + 1) / 2.0

---
## Step 52 — plot images

In [ ]:
for i in range(100):

---
## Step 53 — define subplot

In [ ]:
pyplot.subplot(10, 10, 1 + i)

---
## Step 54 — turn off axis

In [ ]:
pyplot.axis('off')

---
## Step 55 — plot raw pixel data

In [ ]:
pyplot.imshow(X[i, :, :, 0], cmap='gray_r')

---
## Step 56 — save plot to file

In [ ]:
filename1 = 'generated_plot_%04d.png' % (step+1)
	pyplot.savefig(filename1)
	pyplot.close()

---
## Step 57 — save the generator model

In [ ]:
filename2 = 'model_%04d.h5' % (step+1)
	g_model.save(filename2)
	print('>Saved: %s and %s' % (filename1, filename2))

---
## Step 58 — train the generator and discriminator

In [ ]:
def train(g_model, d_model, gan_model, dataset, latent_dim, n_epochs=100, n_batch=64):

---
## Step 59 — calculate the number of batches per training epoch

In [ ]:
bat_per_epo = int(dataset[0].shape[0] / n_batch)

---
## Step 60 — calculate the number of training iterations

In [ ]:
n_steps = bat_per_epo * n_epochs

---
## Step 61 — calculate the size of half a batch of samples

In [ ]:
half_batch = int(n_batch / 2)

---
## Step 62 — manually enumerate epochs

In [ ]:
for i in range(n_steps):

---
## Step 63 — get randomly selected 'real' samples

In [ ]:
[X_real, labels_real], y_real = generate_real_samples(dataset, half_batch)

---
## Step 64 — update discriminator model weights

In [ ]:
_,d_r1,d_r2 = d_model.train_on_batch(X_real, [y_real, labels_real])

---
## Step 65 — generate 'fake' examples

In [ ]:
[X_fake, labels_fake], y_fake = generate_fake_samples(g_model, latent_dim, half_batch)

---
## Step 66 — update discriminator model weights

In [ ]:
_,d_f,d_f2 = d_model.train_on_batch(X_fake, [y_fake, labels_fake])

---
## Step 67 — prepare points in latent space as input for the generator

In [ ]:
[z_input, z_labels] = generate_latent_points(latent_dim, n_batch)

---
## Step 68 — create inverted labels for the fake samples

In [ ]:
y_gan = ones((n_batch, 1))

---
## Step 69 — update the generator via the discriminator's error

In [ ]:
_,g_1,g_2 = gan_model.train_on_batch([z_input, z_labels], [y_gan, z_labels])

---
## Step 70 — summarize loss on this batch

In [ ]:
print('>%d, dr[%.3f,%.3f], df[%.3f,%.3f], g[%.3f,%.3f]' % (i+1, d_r1,d_r2, d_f,d_f2, g_1,g_2))

---
## Step 71 — evaluate the model performance every 'epoch'

In [ ]:
if (i+1) % (bat_per_epo * 10) == 0:
			summarize_performance(i, g_model, latent_dim)

---
## Step 72 — size of the latent space

In [ ]:
latent_dim = 100

---
## Step 73 — create the discriminator

In [ ]:
discriminator = define_discriminator()

---
## Step 74 — create the generator

In [ ]:
generator = define_generator(latent_dim)

---
## Step 75 — create the gan

In [ ]:
gan_model = define_gan(generator, discriminator)

---
## Step 76 — load image data

In [ ]:
dataset = load_real_samples()

---
## Step 77 — train model

In [ ]:
train(generator, discriminator, gan_model, dataset, latent_dim)

---
## Learning Notes / 学习笔记

- **概念**: example of fitting an auxiliary classifier gan (ac-gan) on fashion mnsit 是机器学习中的常用技术。  
  *example of fitting an auxiliary classifier gan (ac-gan) on fashion mnsit is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Train Acgan / 生成对抗网络
# Complete Code / 完整代码
# ===============================

# example of fitting an auxiliary classifier gan (ac-gan) on fashion mnsit
from numpy import zeros
from numpy import ones
from numpy import expand_dims
from numpy.random import randn
from numpy.random import randint
from keras.datasets.fashion_mnist import load_data
from keras.optimizers import Adam
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Reshape
from keras.layers import Flatten
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.layers import BatchNormalization
from keras.layers import Dropout
from keras.layers import Embedding
from keras.layers import Activation
from keras.layers import Concatenate
from keras.initializers import RandomNormal
from matplotlib import pyplot

# define the standalone discriminator model
def define_discriminator(in_shape=(28,28,1), n_classes=10):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# image input
	in_image = Input(shape=in_shape)
	# downsample to 14x14
	fe = Conv2D(32, (3,3), strides=(2,2), padding='same', kernel_initializer=init)(in_image)
	fe = LeakyReLU(alpha=0.2)(fe)
	fe = Dropout(0.5)(fe)
	# normal
	fe = Conv2D(64, (3,3), padding='same', kernel_initializer=init)(fe)
	fe = BatchNormalization()(fe)
	fe = LeakyReLU(alpha=0.2)(fe)
	fe = Dropout(0.5)(fe)
	# downsample to 7x7
	fe = Conv2D(128, (3,3), strides=(2,2), padding='same', kernel_initializer=init)(fe)
	fe = BatchNormalization()(fe)
	fe = LeakyReLU(alpha=0.2)(fe)
	fe = Dropout(0.5)(fe)
	# normal
	fe = Conv2D(256, (3,3), padding='same', kernel_initializer=init)(fe)
	fe = BatchNormalization()(fe)
	fe = LeakyReLU(alpha=0.2)(fe)
	fe = Dropout(0.5)(fe)
	# flatten feature maps
	fe = Flatten()(fe)
	# real/fake output
	out1 = Dense(1, activation='sigmoid')(fe)
	# class label output
	out2 = Dense(n_classes, activation='softmax')(fe)
	# define model
	model = Model(in_image, [out1, out2])
	# compile model
	opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss=['binary_crossentropy', 'sparse_categorical_crossentropy'], optimizer=opt)
	return model

# define the standalone generator model
def define_generator(latent_dim, n_classes=10):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# label input
	in_label = Input(shape=(1,))
	# embedding for categorical input
	li = Embedding(n_classes, 50)(in_label)
	# linear multiplication
	n_nodes = 7 * 7
	li = Dense(n_nodes, kernel_initializer=init)(li)
	# reshape to additional channel
	li = Reshape((7, 7, 1))(li)
	# image generator input
	in_lat = Input(shape=(latent_dim,))
	# foundation for 7x7 image
	n_nodes = 384 * 7 * 7
	gen = Dense(n_nodes, kernel_initializer=init)(in_lat)
	gen = Activation('relu')(gen)
	gen = Reshape((7, 7, 384))(gen)
	# merge image gen and label input
	merge = Concatenate()([gen, li])
	# upsample to 14x14
	gen = Conv2DTranspose(192, (5,5), strides=(2,2), padding='same', kernel_initializer=init)(merge)
	gen = BatchNormalization()(gen)
	gen = Activation('relu')(gen)
	# upsample to 28x28
	gen = Conv2DTranspose(1, (5,5), strides=(2,2), padding='same', kernel_initializer=init)(gen)
	out_layer = Activation('tanh')(gen)
	# define model
	model = Model([in_lat, in_label], out_layer)
	return model

# define the combined generator and discriminator model, for updating the generator
def define_gan(g_model, d_model):
	# make weights in the discriminator not trainable
	for layer in d_model.layers:
		if not isinstance(layer, BatchNormalization):
			layer.trainable = False
	# connect the outputs of the generator to the inputs of the discriminator
	gan_output = d_model(g_model.output)
	# define gan model as taking noise and label and outputting real/fake and label outputs
	model = Model(g_model.input, gan_output)
	# compile model
	opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss=['binary_crossentropy', 'sparse_categorical_crossentropy'], optimizer=opt)
	return model

# load images
def load_real_samples():
	# load dataset
	(trainX, trainy), (_, _) = load_data()
	# expand to 3d, e.g. add channels
	X = expand_dims(trainX, axis=-1)
	# convert from ints to floats
	X = X.astype('float32')
	# scale from [0,255] to [-1,1]
	X = (X - 127.5) / 127.5
	print(X.shape, trainy.shape)
	return [X, trainy]

# select real samples
def generate_real_samples(dataset, n_samples):
	# split into images and labels
	images, labels = dataset
	# choose random instances
	ix = randint(0, images.shape[0], n_samples)
	# select images and labels
	X, labels = images[ix], labels[ix]
	# generate class labels
	y = ones((n_samples, 1))
	return [X, labels], y

# generate points in latent space as input for the generator
def generate_latent_points(latent_dim, n_samples, n_classes=10):
	# generate points in the latent space
	x_input = randn(latent_dim * n_samples)
	# reshape into a batch of inputs for the network
	z_input = x_input.reshape(n_samples, latent_dim)
	# generate labels
	labels = randint(0, n_classes, n_samples)
	return [z_input, labels]

# use the generator to generate n fake examples, with class labels
def generate_fake_samples(generator, latent_dim, n_samples):
	# generate points in latent space
	z_input, labels_input = generate_latent_points(latent_dim, n_samples)
	# predict outputs
	images = generator.predict([z_input, labels_input])
	# create class labels
	y = zeros((n_samples, 1))
	return [images, labels_input], y

# generate samples and save as a plot and save the model
def summarize_performance(step, g_model, latent_dim, n_samples=100):
	# prepare fake examples
	[X, _], _ = generate_fake_samples(g_model, latent_dim, n_samples)
	# scale from [-1,1] to [0,1]
	X = (X + 1) / 2.0
	# plot images
	for i in range(100):
		# define subplot
		pyplot.subplot(10, 10, 1 + i)
		# turn off axis
		pyplot.axis('off')
		# plot raw pixel data
		pyplot.imshow(X[i, :, :, 0], cmap='gray_r')
	# save plot to file
	filename1 = 'generated_plot_%04d.png' % (step+1)
	pyplot.savefig(filename1)
	pyplot.close()
	# save the generator model
	filename2 = 'model_%04d.h5' % (step+1)
	g_model.save(filename2)
	print('>Saved: %s and %s' % (filename1, filename2))

# train the generator and discriminator
def train(g_model, d_model, gan_model, dataset, latent_dim, n_epochs=100, n_batch=64):
	# calculate the number of batches per training epoch
	bat_per_epo = int(dataset[0].shape[0] / n_batch)
	# calculate the number of training iterations
	n_steps = bat_per_epo * n_epochs
	# calculate the size of half a batch of samples
	half_batch = int(n_batch / 2)
	# manually enumerate epochs
	for i in range(n_steps):
		# get randomly selected 'real' samples
		[X_real, labels_real], y_real = generate_real_samples(dataset, half_batch)
		# update discriminator model weights
		_,d_r1,d_r2 = d_model.train_on_batch(X_real, [y_real, labels_real])
		# generate 'fake' examples
		[X_fake, labels_fake], y_fake = generate_fake_samples(g_model, latent_dim, half_batch)
		# update discriminator model weights
		_,d_f,d_f2 = d_model.train_on_batch(X_fake, [y_fake, labels_fake])
		# prepare points in latent space as input for the generator
		[z_input, z_labels] = generate_latent_points(latent_dim, n_batch)
		# create inverted labels for the fake samples
		y_gan = ones((n_batch, 1))
		# update the generator via the discriminator's error
		_,g_1,g_2 = gan_model.train_on_batch([z_input, z_labels], [y_gan, z_labels])
		# summarize loss on this batch
		print('>%d, dr[%.3f,%.3f], df[%.3f,%.3f], g[%.3f,%.3f]' % (i+1, d_r1,d_r2, d_f,d_f2, g_1,g_2))
		# evaluate the model performance every 'epoch'
		if (i+1) % (bat_per_epo * 10) == 0:
			summarize_performance(i, g_model, latent_dim)

# size of the latent space
latent_dim = 100
# create the discriminator
discriminator = define_discriminator()
# create the generator
generator = define_generator(latent_dim)
# create the gan
gan_model = define_gan(generator, discriminator)
# load image data
dataset = load_real_samples()
# train model
train(generator, discriminator, gan_model, dataset, latent_dim)

---

➡️ **Next / 下一步**: File 4 of 4